<a href="https://colab.research.google.com/github/Inventiv-PH/subscriber-sim/blob/main/subscriber_sim_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!/usr/bin/env python3
"""
════════════════════════════════════════════════════════════════════════════
IMPROVED SUBSCRIBER SIMULATOR
A production-ready implementation with optimized accuracy & naturalness
════════════════════════════════════════════════════════════════════════════

Features:
- 7 distinct personality archetypes with character-driven prompts
- Archetype-specific generation parameters for consistency
- Response filtering to prevent OOC (out of character) breaks
- Smart context windowing for fresh, relevant responses
- Multi-turn conversation support with history management
- Gradio UI for interactive testing
- Session saving to JSONL for training data collection

Usage:
    python subscriber_sim_improved.py --mode inference
    python subscriber_sim_improved.py --mode collect_data
"""

import sys
import os
import re
import json
import uuid
import torch
import argparse
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, asdict

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIG & CONSTANTS
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    """Configuration for subscriber simulator."""
    model_name: str = 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit'
    max_seq_length: int = 2048
    load_in_4bit: bool = True
    data_dir: Path = Path('./data')
    is_colab: bool = False
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

    def __post_init__(self):
        self.data_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# ARCHETYPE DEFINITIONS (CHARACTER-DRIVEN)
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class Archetype:
    """Definition of a subscriber personality archetype."""
    label: str
    system_prompt: str
    opener: str


ARCHETYPES: Dict[str, Archetype] = {
    'horny': Archetype(
        label='🔥 Horny',
        system_prompt="""You're a 22-year-old guy flirting with a hot creator on OnlyFans.
You're into trans women and not shy about it. When you like something,
you're direct and sexual. You text like a normal guy—quick messages, emojis,
no overthinking. You banter, tease, and aren't afraid to ask for what you want
(or what she's selling). Natural texting vibes.""",
        opener='hey sexy 😏 saw ur page and damn... u got me hard already',
    ),

    'cheapskate': Archetype(
        label='💸 Cheapskate',
        system_prompt="""You're a broke dude on OnlyFans who loves content but hates
spending money. You try to get freebies, negotiate prices, or watch free teasers.
You're not mean about it—just cheap and clever. Sometimes you flirt to lower the
price or get custom stuff discounted. You test boundaries but aren't a jerk.""",
        opener='yo ur page is fire but those prices tho 👀 any deals?',
    ),

    'casual': Archetype(
        label='😎 Casual',
        system_prompt="""You're a casual fan who appreciates the content.
Not obsessed, not cheap—just here because you like her vibe.
You chat naturally, maybe tip occasionally, and support her without drama.
You might ask questions, compliment genuinely, or just hang out in the comments.""",
        opener='hey! loving the content on here, keep it up 👍',
    ),

    'troll': Archetype(
        label='😈 Troll',
        system_prompt="""You're here to mess with creators—it's funny to you.
You make inappropriate jokes, ask for free stuff, try to get a reaction.
You're not genuinely here for the content; you're here for the chaos.
You back off if someone's clearly upset but keep probing to see what you can get away with.""",
        opener='lol ur prices are insane 💀 can i get a freebie to test the goods first',
    ),

    'whale': Archetype(
        label='🐋 Whale',
        system_prompt="""You have money and love spending it on creators you like.
You don't need to negotiate—you just buy exclusive content and tip generously.
You're respectful, enthusiastic, and get excited about her releases.
You like feeling special and don't mind paying for that.""",
        opener='hey gorgeous, your page is amazing. just bought your full collection 💳✨',
    ),

    'cold': Archetype(
        label='❄️ Cold',
        system_prompt="""You're not really engaged. You're here because a friend
recommended the page or you were bored. You're polite but distant.
You might leave or stick around depending on if something grabs your attention.
You're not rude, just not invested. Your messages are short, neutral.""",
        opener='ok i subscribed. what now.',
    ),

    'simp': Archetype(
        label='💕 Simp',
        system_prompt="""You're genuinely in love/obsessed with this creator.
Every message is about how beautiful/talented/perfect she is.
You drop money freely, defend her against everyone, and desperately want
her to notice you as special. You're earnest, maybe a little delusional,
but not in a dangerous way—just hopelessly infatuated.""",
        opener='omg you are literally the most gorgeous person i have ever seen 😍😍',
    ),
}


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GENERATION PARAMETERS (ARCHETYPE-SPECIFIC)
# ════════════════════════════════════════════════════════════════════════════

@dataclass
class GenerationParams:
    """Parameters for text generation per archetype."""
    temperature: float
    top_p: float
    max_new_tokens: int
    repetition_penalty: float


GENERATION_PARAMS: Dict[str, GenerationParams] = {
    'horny': GenerationParams(
        temperature=0.75,
        top_p=0.85,
        max_new_tokens=80,
        repetition_penalty=1.05,
    ),
    'cheapskate': GenerationParams(
        temperature=0.70,
        top_p=0.80,
        max_new_tokens=100,
        repetition_penalty=1.0,
    ),
    'whale': GenerationParams(
        temperature=0.65,
        top_p=0.75,
        max_new_tokens=90,
        repetition_penalty=1.05,
    ),
    'troll': GenerationParams(
        temperature=0.80,
        top_p=0.90,
        max_new_tokens=85,
        repetition_penalty=1.15,
    ),
    'simp': GenerationParams(
        temperature=0.75,
        top_p=0.85,
        max_new_tokens=95,
        repetition_penalty=1.0,
    ),
    'casual': GenerationParams(
        temperature=0.65,
        top_p=0.80,
        max_new_tokens=75,
        repetition_penalty=1.0,
    ),
    'cold': GenerationParams(
        temperature=0.60,
        top_p=0.70,
        max_new_tokens=60,
        repetition_penalty=1.0,
    ),
}

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# RESPONSE FILTERING & POST-PROCESSING
# ════════════════════════════════════════════════════════════════════════════

class ResponseFilter:
    """Filters and post-processes model responses for character consistency."""

    # Regex patterns for AI meta-commentary
    META_PATTERNS = [
        r"(?:I'm|I am) (?:an|a) (?:AI|bot|model|language model|assistant)",
        r"(?:As an|As a) (?:AI|bot|model|language model|assistant)",
        r"I (?:should|shouldn't|need to|must|cannot|can't) ",
        r"I (?:can't|cannot) (?:roleplay|pretend)",
        r"^(?:I understand|I realize|I appreciate that)",
        r"^(?:Let me|I'll|I would) roleplay",
        r"I (?:must )?(?:maintain|keep) (?:appropriate|professional)",
    ]

    # Max sentences per archetype
    MAX_SENTENCES = {
        'horny': 3, 'cheapskate': 3, 'casual': 2,
        'troll': 2, 'whale': 2, 'simp': 3, 'cold': 1
    }

    @classmethod
    def filter(cls, response: str, archetype_key: str) -> str:
        """Post-process response to ensure character consistency."""

        if not response or not response.strip():
            return response

        filtered = response.strip()

        # 1. Remove meta-commentary & AI self-reference
        for pattern in cls.META_PATTERNS:
            filtered = re.sub(pattern, "", filtered, flags=re.IGNORECASE)

        filtered = filtered.strip()

        # 2. Remove leading/trailing quotes (if parroting)
        if (filtered.startswith('"') and filtered.endswith('"')) or \
           (filtered.startswith("'") and filtered.endswith("'")):
            filtered = filtered[1:-1]

        filtered = filtered.strip()

        # 3. Enforce archetype-specific max sentence count
        max_sents = cls.MAX_SENTENCES.get(archetype_key, 2)
        sentences = [s.strip() for s in filtered.split('. ')]
        sentences = [s for s in sentences if s]

        if len(sentences) > max_sents:
            filtered = '. '.join(sentences[:max_sents])
            if not filtered.endswith(('.', '!', '?')):
                filtered += '.'

        filtered = filtered.strip()

        # 4. Reduce excessive punctuation
        filtered = re.sub(r'([!?.♥💦🔥])\1{2,}', r'\1\1', filtered)
        filtered = re.sub(r'\.\.\.+', '..', filtered)

        # 5. Remove leading asterisks (action formatting)
        filtered = re.sub(r'^\*+\s*', '', filtered)

        # 6. Fix ALL CAPS words (except short ones like "OK", "LOL")
        words = filtered.split()
        for i, word in enumerate(words):
            if word.isupper() and len(word) > 3:
                words[i] = word.capitalize()
        filtered = ' '.join(words)

        filtered = filtered.strip()

        # 7. Ensure non-empty response
        if not filtered:
            filtered = response.strip()

        return filtered

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MODEL MANAGEMENT
# ════════════════════════════════════════════════════════════════════════════

class SubscriberModel:
    """Loads and manages the subscriber simulator model."""

    def __init__(self, config: Config):
        self.config = config
        self.model = None
        self.tokenizer = None
        self._load_model()

    def _load_model(self):
        """Load model and tokenizer."""
        try:
            if 'google.colab' in sys.modules:
                from unsloth import FastLanguageModel
                from unsloth.chat_templates import get_chat_template

                print(f"Loading {self.config.model_name} for Colab...")
                self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                    model_name=self.config.model_name,
                    max_seq_length=self.config.max_seq_length,
                    dtype=None,
                    load_in_4bit=self.config.load_in_4bit,
                )
                self.tokenizer = get_chat_template(self.tokenizer, chat_template='llama-3.1')
                FastLanguageModel.for_inference(self.model)
            else:
                from transformers import AutoModelForCausalLM, AutoTokenizer

                print(f"Loading {self.config.model_name} locally...")
                self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
                self.model = AutoModelForCausalLM.from_pretrained(
                    self.config.model_name,
                    torch_dtype=torch.float16,
                    device_map='auto',
                )
                self.model.eval()

        except Exception as e:
            print(f"Error loading model: {e}")
            print("Make sure you have the required libraries installed:")
            print("  pip install torch transformers unsloth")
            raise

    def generate(self, messages: List[Dict[str, str]], archetype_key: str) -> str:
        """Generate a response given messages and archetype."""

        params = GENERATION_PARAMS[archetype_key]

        # Apply chat template
        inputs = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors='pt',
        )

        # Handle various input formats
        if hasattr(inputs, 'input_ids'):
            input_tensor = inputs.input_ids
        else:
            input_tensor = inputs

        input_tensor = input_tensor.to(self.config.device)
        input_length = input_tensor.shape[1]

        # Generate
        with torch.inference_mode():
            outputs = self.model.generate(
                input_ids=input_tensor,
                max_new_tokens=params.max_new_tokens,
                temperature=params.temperature,
                top_p=params.top_p,
                do_sample=True,
                repetition_penalty=params.repetition_penalty,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )

        # Decode
        response = self.tokenizer.decode(
            outputs[0][input_length:],
            skip_special_tokens=True,
        ).strip()

        # Clean up memory
        del inputs, input_tensor, outputs
        torch.cuda.empty_cache()

        # Filter for character consistency
        response = ResponseFilter.filter(response, archetype_key)

        return response

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# CONVERSATION MANAGER
# ════════════════════════════════════════════════════════════════════════════

class ConversationManager:
    """Manages multi-turn conversations with context windowing."""

    MAX_HISTORY_TURNS = 6  # Keep last 3 exchanges

    def __init__(self, archetype_key: str):
        self.archetype_key = archetype_key
        self.history: List[Dict[str, str]] = []
        self.session_id = str(uuid.uuid4())[:8]

    def add_message(self, role: str, content: str):
        """Add a message to conversation history."""
        self.history.append({'role': role, 'content': content})

    def get_windowed_history(self) -> List[Dict[str, str]]:
        """Get conversation history with context windowing."""
        if len(self.history) > self.MAX_HISTORY_TURNS:
            return self.history[-self.MAX_HISTORY_TURNS:]
        return self.history

    def build_messages(self, new_user_message: str) -> List[Dict[str, str]]:
        """Build message list for model input."""
        archetype = ARCHETYPES[self.archetype_key]

        messages = [{"role": "system", "content": archetype.system_prompt}]

        # Add windowed history
        for msg in self.get_windowed_history():
            messages.append(msg)

        # Add new user message
        messages.append({"role": "user", "content": new_user_message})

        return messages

    def reset(self):
        """Reset conversation for new session."""
        self.history = []
        self.session_id = str(uuid.uuid4())[:8]

    def to_dict(self) -> Dict:
        """Serialize conversation to dict."""
        return {
            'session_id': self.session_id,
            'archetype': self.archetype_key,
            'timestamp': datetime.now().isoformat(),
            'messages': self.history,
        }


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SUBSCRIBER SIMULATOR
# ════════════════════════════════════════════════════════════════════════════

class SubscriberSimulator:
    """Main simulator class."""

    def __init__(self, config: Config):
        self.config = config
        self.model = SubscriberModel(config)
        self.conversations: Dict[str, ConversationManager] = {
            arch_key: ConversationManager(arch_key)
            for arch_key in ARCHETYPES.keys()
        }

    def generate_response(self, user_message: str, archetype_key: str) -> str:
        """Generate a subscriber response."""

        if archetype_key not in ARCHETYPES:
            raise ValueError(f"Unknown archetype: {archetype_key}")

        # Get or create conversation
        conv = self.conversations[archetype_key]

        # Build messages with context
        messages = conv.build_messages(user_message)

        # Generate response
        response = self.model.generate(messages, archetype_key)

        # Update conversation history
        conv.add_message("user", user_message)
        conv.add_message("assistant", response)

        return response

    def start_new_session(self, archetype_key: str) -> str:
        """Start a new conversation session."""
        if archetype_key not in ARCHETYPES:
            raise ValueError(f"Unknown archetype: {archetype_key}")

        self.conversations[archetype_key].reset()
        return ARCHETYPES[archetype_key].opener

    def save_session(self, archetype_key: str) -> Path:
        """Save conversation to JSONL file."""
        conv = self.conversations[archetype_key]
        data = conv.to_dict()

        # Save to file
        filename = f"{archetype_key}_{data['session_id']}_{int(datetime.now().timestamp())}.jsonl"
        filepath = self.config.data_dir / filename

        with open(filepath, 'w') as f:
            f.write(json.dumps(data) + '\n')

        return filepath

    def test_all_archetypes(self, test_message: str = "hey babe 😘") -> Dict[str, str]:
        """Test all archetypes with same message."""
        results = {}

        for arch_key in ARCHETYPES.keys():
            self.start_new_session(arch_key)
            response = self.generate_response(test_message, arch_key)
            results[arch_key] = response

        return results

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GRADIO UI (OPTIONAL)
# ════════════════════════════════════════════════════════════════════════════

def create_gradio_ui(simulator: SubscriberSimulator):
    """Create interactive Gradio chat UI."""

    try:
        import gradio as gr
    except ImportError:
        print("Gradio not installed. Install with: pip install gradio")
        return

    current_archetype = 'horny'

    def on_archetype_change(archetype_key):
        nonlocal current_archetype
        current_archetype = archetype_key
        opener = simulator.start_new_session(archetype_key)
        return [{'role': 'assistant', 'content': opener}]

    def on_message_submit(user_message, history):
        nonlocal current_archetype

        if not user_message.strip():
            return "", history

        response = simulator.generate_response(user_message, current_archetype)

        history.append({'role': 'user', 'content': user_message})
        history.append({'role': 'assistant', 'content': response})

        return "", history

    def on_save_session():
        filepath = simulator.save_session(current_archetype)
        return f"✅ Conversation saved to {filepath.name}"

    def on_new_session(archetype_key):
        nonlocal current_archetype
        current_archetype = archetype_key
        opener = simulator.start_new_session(archetype_key)
        return [{'role': 'assistant', 'content': opener}], ""

    # UI Layout
    archetype_choices = [(v.label, k) for k, v in ARCHETYPES.items()]

    with gr.Blocks(title='Subscriber Simulator', theme=gr.themes.Soft()) as demo:
        gr.Markdown('# 🎭 Subscriber Simulator\nYou are **Jasmin**. The bot is a subscriber.')

        with gr.Row():
            with gr.Column(scale=2):
                archetype_dropdown = gr.Dropdown(
                    choices=archetype_choices,
                    value='horny',
                    label='Subscriber Archetype',
                )
            with gr.Column(scale=1):
                new_session_btn = gr.Button('🆕 New Session', scale=1)
                save_btn = gr.Button('💾 Save', variant='primary', scale=1)

        chatbot = gr.Chatbot(
            value=[{'role': 'assistant', 'content': ARCHETYPES['horny'].opener}],
            label='Conversation',
            height=500,
            type='messages',
        )

        msg_input = gr.Textbox(
            placeholder='Type your message as Jasmin...',
            label='Your message',
            show_label=False,
        )

        status_output = gr.Textbox(label='Status', interactive=False)

        # Event handlers
        archetype_dropdown.change(
            fn=on_archetype_change,
            inputs=[archetype_dropdown],
            outputs=[chatbot],
        )

        msg_input.submit(
            fn=on_message_submit,
            inputs=[msg_input, chatbot],
            outputs=[msg_input, chatbot],
        )

        save_btn.click(
            fn=on_save_session,
            inputs=[],
            outputs=[status_output],
        )

        new_session_btn.click(
            fn=on_new_session,
            inputs=[archetype_dropdown],
            outputs=[chatbot, msg_input],
        )

    return demo

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# COMMAND LINE INTERFACE
# ════════════════════════════════════════════════════════════════════════════

def run_cli_mode(simulator: SubscriberSimulator, archetype_key: str = 'horny'):
    """Run interactive CLI mode."""

    print(f"\n{'='*60}")
    print(f"🎭 Subscriber Simulator - CLI Mode")
    print(f"{'='*60}\n")

    print(f"Selected: {ARCHETYPES[archetype_key].label}")

    simulator.start_new_session(archetype_key)
    opener = ARCHETYPES[archetype_key].opener
    print(f"\nSubscriber: {opener}\n")

    try:
        while True:
            user_input = input("You: ").strip()

            if not user_input:
                continue

            if user_input.lower() in ['quit', 'exit', 'q']:
                print("\nGoodbye! 👋")
                break

            if user_input.lower().startswith('/archetype '):
                new_arch = user_input.split(' ', 1)[1]
                if new_arch in ARCHETYPES:
                    archetype_key = new_arch
                    simulator.start_new_session(archetype_key)
                    print(f"\nSwitched to: {ARCHETYPES[archetype_key].label}")
                    print(f"Opener: {ARCHETYPES[archetype_key].opener}\n")
                else:
                    print(f"Unknown archetype: {new_arch}")
                continue

            if user_input.lower() == '/save':
                path = simulator.save_session(archetype_key)
                print(f"✅ Saved to {path}\n")
                continue

            if user_input.lower() == '/new':
                simulator.start_new_session(archetype_key)
                print(f"New session started. Opener: {ARCHETYPES[archetype_key].opener}\n")
                continue

            if user_input.lower() == '/help':
                print("""
Commands:
  /archetype [name]  - Switch archetype (horny, cheapskate, casual, troll, whale, cold, simp)
  /save              - Save conversation
  /new               - Start new session
  /help              - Show this help
  quit/exit/q        - Exit
                """)
                continue

            response = simulator.generate_response(user_input, archetype_key)
            print(f"\nSubscriber: {response}\n")

    except KeyboardInterrupt:
        print("\n\nGoodbye! 👋")


def run_test_mode(simulator: SubscriberSimulator):
    """Run test mode."""

    test_messages = [
        "hey babe welcome to my page 😘",
        "wanna see something special? i got a new vid just for you 🔥",
        "that'll be $25 for the full set baby",
        "you're so sweet, thanks for subscribing ❤️",
    ]

    print(f"\n{'='*60}")
    print(f"🧪 Testing All Archetypes")
    print(f"{'='*60}\n")

    for test_msg in test_messages:
        print(f"📝 Message: \"{test_msg}\"\n")

        results = simulator.test_all_archetypes(test_msg)

        for arch_key, response in results.items():
            label = ARCHETYPES[arch_key].label
            print(f"  {label:15} → {response}")

        print()



In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════

def main():
    parser = argparse.ArgumentParser(
        description='Subscriber Simulator - Generate realistic OnlyFans subscriber messages'
    )
    parser.add_argument(
        '--mode',
        choices=['cli', 'gradio', 'test'],
        default='cli',
        help='Run mode (default: cli)',
    )
    parser.add_argument(
        '--archetype',
        choices=list(ARCHETYPES.keys()),
        default='horny',
        help='Starting archetype (default: horny)',
    )
    parser.add_argument(
        '--data-dir',
        type=Path,
        default=Path('./data'),
        help='Data directory (default: ./data)',
    )
    parser.add_argument(
        '--no-4bit',
        action='store_true',
        help='Disable 4-bit quantization',
    )

    args = parser.parse_args()

    # Create config
    config = Config(
        data_dir=args.data_dir,
        load_in_4bit=not args.no_4bit,
        is_colab='google.colab' in sys.modules,
    )

    # Create simulator
    print("Initializing Subscriber Simulator...")
    simulator = SubscriberSimulator(config)

    # Run selected mode
    if args.mode == 'cli':
        run_cli_mode(simulator, args.archetype)

    elif args.mode == 'test':
        run_test_mode(simulator)

    elif args.mode == 'gradio':
        print("\nLaunching Gradio UI...")
        demo = create_gradio_ui(simulator)
        demo.launch(share=config.is_colab, quiet=True)


if __name__ == '__main__':
    main()